**General formula to implement a Recurrent Neural Network**

`` h<sub>t</sub> = tanh( x<sub>t</sub> + h<sub>t-1</sub> )``

### Sequence to Sequence Models:
- Many to one - *Classification*
- Many to many - *text generation*
- Many to many - *neural machine translation*


### **RNNs (Recurrent Neural Networks)**

* RNNs are for **sequential data** (text, speech, time series).

* Reads input **one word at a time**, maintaining a **hidden state (memory)**.

* Hidden state `hₜ` combines:

  * current word vector `xₜ`
  * previous hidden state `hₜ₋₁`
    → `hₜ = tanh(Wₓ*xₜ + Wₕ*hₜ₋₁)`

* Initial hidden state `h₀` is usually **all zeros**.

* Weight matrices `Wₓ` and `Wₕ` are **learned during training** using many sequences.

---

### **How RNN predicts words**

* At each step, the RNN can predict the **next word** using `hₜ`.
* Prediction = `softmax(V * hₜ)` → probabilities for all words in vocabulary.
* Word with **highest probability** is chosen as next word.

---

### **Memory + Word Vectors**

* Word embeddings (`xₜ`) encode **semantic meaning**.
* Hidden state (`hₜ`) encodes **context so far**.
* Together, they **allow the RNN to predict the next word** accurately.

---

### **Training / Iteration**

* Single sentence → inner loop over words (updates `hₜ`).
* Training dataset → outer loop over sentences.
* Backpropagation Through Time (BPTT) updates **all weights** to minimize prediction error.
* Learning is **iterative**, over many sentences, for the network to capture grammar & meaning.



In [2]:
import numpy as np

vocab = ["I","love","machine","learning"]
vocab_size = len(vocab)
vocab_size

4

In [3]:
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i,word in enumerate(vocab)}
word_to_idx, idx_to_word

({'I': 0, 'love': 1, 'machine': 2, 'learning': 3},
 {0: 'I', 1: 'love', 2: 'machine', 3: 'learning'})

In [4]:
## One hot encoding.
def one_hot(word_idx):
    vec = np.zeros((vocab_size,1))
    vec[word_idx] = 1
    return vec


In [6]:
# Dimensions
input_size = vocab_size       # size of one-hot vector
hidden_size = 5               # small hidden layer
output_size = vocab_size      # same as vocab for predicting next word

# Weight matrices
Wx = np.random.randn(hidden_size, input_size) * 0.01  # input → hidden
Wh = np.random.randn(hidden_size, hidden_size) * 0.01 # hidden → hidden
Wy = np.random.randn(output_size, hidden_size) * 0.01 # hidden → output

# Hidden state initialization
h0 = np.zeros((hidden_size, 1))


In [7]:
def tanh(x):
    return np.tanh(x)

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0, keepdims=True)


In [17]:
# Example sequence: "I love machine"
sequence = ["I", "love", "machine"] # tokenization

h = h0 # intial value assignment to memory
print("Forward pass:")
for t, word in enumerate(sequence):
    x_t = one_hot(word_to_idx[word])        # current word vector 
    h = tanh(np.dot(Wx, x_t) + np.dot(Wh, h))  # update hidden state
    y = softmax(np.dot(Wy, h))                  # predicted next word probabilities
    
    print(f"t={t}, word='{word}'")
    print(f"Hidden state h:\n{h.flatten()}")
    print(f"Predicted probs:\n{y.flatten()}\n")
    # print(x_t, end="\n\n")


Forward pass:
t=0, word='I'
Hidden state h:
[-0.00510686  0.00716994  0.00297507 -0.00796233 -0.00685505]
Predicted probs:
[0.25005559 0.24996909 0.24998779 0.24998753]

t=1, word='love'
Hidden state h:
[ 0.00039488  0.01284667  0.00159018 -0.01798432  0.01023198]
Predicted probs:
[0.25002515 0.24995386 0.25002234 0.24999865]

t=2, word='machine'
Hidden state h:
[-0.00470639  0.01021312  0.00289708  0.00830132 -0.01861416]
Predicted probs:
[0.2500735  0.24996363 0.24999836 0.24996451]



In [18]:
target_word = "learning"
target_idx = word_to_idx[target_word]


In [25]:
learning_rate = 0.1

for epoch in range(1000):

    # ---- forward ----
    xs, hs, ys = {}, {}, {}
    hs[-1] = h0

    for t, word in enumerate(sequence):
        xs[t] = one_hot(word_to_idx[word])
        hs[t] = np.tanh(Wx @ xs[t] + Wh @ hs[t-1])
        ys[t] = softmax(Wy @ hs[t])

    # compute loss (LAST timestep only)
    loss = -np.log(ys[len(sequence)-1][target_idx, 0])

    # ---- backward (BPTT) ----
    dWx = np.zeros_like(Wx)
    dWh = np.zeros_like(Wh)
    dWy = np.zeros_like(Wy)
    dh_next = np.zeros_like(h0)

    # output gradient
    dy = ys[len(sequence)-1].copy()
    dy[target_idx] -= 1

    dWy += dy @ hs[len(sequence)-1].T
    dh = Wy.T @ dy

    # backprop through time
    for t in reversed(range(len(sequence))):
        dh = dh + dh_next
        dtanh = (1 - hs[t]**2) * dh

        dWx += dtanh @ xs[t].T
        dWh += dtanh @ hs[t-1].T

        dh_next = Wh.T @ dtanh

    # ---- update ----
    Wx -= learning_rate * dWx
    Wh -= learning_rate * dWh
    Wy -= learning_rate * dWy

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")


Epoch 0, Loss: 1.3863
Epoch 100, Loss: 0.0434
Epoch 200, Loss: 0.0121
Epoch 300, Loss: 0.0068
Epoch 400, Loss: 0.0047
Epoch 500, Loss: 0.0036
Epoch 600, Loss: 0.0029
Epoch 700, Loss: 0.0024
Epoch 800, Loss: 0.0021
Epoch 900, Loss: 0.0018


In [26]:
predicted_idx = np.argmax(ys[len(sequence)-1])
print("Predicted next word:", idx_to_word[predicted_idx])


Predicted next word: learning
